# 15 - Create the project group and preview Azure

Create one project Resource Group, then validate and preview infra/platform.json with Azure. The template plans Storage, a private Blob container, Basic ACR, shared logs, Key Vault, Application Insights, Azure ML workspace, and a Consumption Container Apps environment. This notebook does not deploy these eight service/storage resources or the three scoped roles for the workspace identity.

The 91.90 THB estimate is still a plan. ACR charges start after paid deployment; storage and logs depend on usage. No training VM, app replica, Dedicated profile, or private endpoint is in this template. The removeAfter tag is a cleanup reminder, not automatic deletion.


In [7]:
from pathlib import Path
import json
import platform
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())

access = json.loads((ROOT / "reports/azure-access-quota.json").read_text())
assert not access["checks_failed"], "Finish 14 first."
template = json.loads((ROOT / "infra/platform.json").read_text())
print("Planned resources:")
for resource in template["resources"]:
    print("-", resource["type"])
print("Next cell creates the project group and previews this template.")


Planned resources:
- Microsoft.Storage/storageAccounts
- Microsoft.Storage/storageAccounts/blobServices/containers
- Microsoft.ContainerRegistry/registries
- Microsoft.OperationalInsights/workspaces
- Microsoft.KeyVault/vaults
- Microsoft.Insights/components
- Microsoft.MachineLearningServices/workspaces
- Microsoft.App/managedEnvironments
- Microsoft.Authorization/roleAssignments
- Microsoft.Authorization/roleAssignments
- Microsoft.Authorization/roleAssignments
Next cell creates the project group and previews this template.


## Create group and preview

Run once. Azure validation and what-if may take several minutes. The script keeps an existing group only if its project tag and region match. It does not delete resources, apply the template, print keys, or change installed extensions.

Expect errors: [], validation_passed: true, and preview_ready: true. Review changes; only Create and NoChange are accepted. If preview fails, save the result and stop. Existing group creation is recorded even if a later check fails.


In [10]:
result = subprocess.run([sys.executable, "-m", "cloudlayer.platform_preview"],
                        cwd=ROOT, capture_output=True, text=True)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Preview script failed. Read the traceback above.")
report = json.loads(result.stdout)
print(json.dumps(report, indent=2))
assert not report["errors"] and report["preview_ready"], "Stop and send the preview report."


{
  "checked_at": "2026-10-11T14:56:26.645728+00:00",
  "resource_group_created": false,
  "paid_resources_created": false,
  "validation_passed": true,
  "preview_ready": true,
  "errors": [],
  "resource_group": "rg-mlop-pj-492ea24d",
  "region": "centralindia",
  "suffix": "492ea24d",
  "project_tag": "mlop-pj-6688012",
  "budget_thb_before_credits": 150,
  "allowed_regions": [
    "centralindia",
    "indiasouthcentral",
    "indonesiacentral",
    "japanwest",
    "malaysiawest"
  ],
  "group_reused": true,
  "resource_group_location": "southeastasia",
  "cleanup_by_utc": "2026-10-13T14:15:40.612068+00:00",
  "template_file": "infra/platform-recovery.json",
  "existing_services_checked": true,
  "template_sha256": "a711b17d0c30bef4631a724a9a1de76a5370ebb84880da42ab6612f808d9c6e6",
  "parameters_sha256": "53685a0d3af0b45a0792db6a8662ed7f5211367e450e93b9bf8929661dd35d32",
  "changes": [
    {
      "change": "Create",
      "resource": "Microsoft.ContainerRegistry/registries/acrmlop

## Output and next step

Save this notebook. Output: reports/azure-platform-preview.json. Local parameters are in tmp/platform-parameters.json. Keep both for the next step. Send the result before deployment; the next notebook applies the reviewed template and starts applicable charges.

[ARM what-if](https://learn.microsoft.com/en-us/azure/azure-resource-manager/templates/deploy-what-if) | [Resource groups](https://learn.microsoft.com/en-us/cli/azure/group?view=azure-cli-latest#az-group-create)
